In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install langchain langchain-community langchain-chroma chromadb langchain-groq

In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

### create a retriever
**Retriever** is a runnable as we use .invoke() func.  

**retriever** use various methods such as semantic search, keyword matching, or hybrid approaches for finding info  
while **semantic** search in vector store uses a fixed method

In [4]:
#source documents - static as of now

semantic_docs = [
    Document(page_content="Global warming is caused by the increase of greenhouse gases in the atmosphere.", metadata={"topic": "climate", "type": "semantic"}),
    Document(page_content="Rising temperatures are a direct result of heat-trapping pollutants in the air.", metadata={"topic": "climate", "type": "semantic"}),
    Document(page_content="The planet is getting hotter because carbon emissions are reaching record levels.", metadata={"topic": "climate", "type": "semantic"}),
    Document(page_content="Climate change is driven by the buildup of CO2 from burning fossil fuels.", metadata={"topic": "climate", "type": "semantic"}),
    Document(page_content="Thermal energy is being retained by the earth's atmosphere due to industrial smoke.", metadata={"topic": "climate", "type": "semantic"})
]

In [6]:
vector_store = Chroma.from_documents(
    documents=semantic_docs,
    embedding=embeddings,
    collection_name='my_collection'
)

In [7]:
retriever = vector_store.as_retriever(search_kwargs={'k':3})  #top 3 results

In [8]:
query = "Causes of global warming"
result = retriever.invoke(query)
# will return results that are similar even if they are almost same

for i,doc in enumerate(result):
    print(f"\n------Result {i+1}------")
    print(doc.page_content)


------Result 1------
Global warming is caused by the increase of greenhouse gases in the atmosphere.

------Result 2------
Climate change is driven by the buildup of CO2 from burning fossil fuels.

------Result 3------
The planet is getting hotter because carbon emissions are reaching record levels.


# Maximum Marginal Relevance MMR
It picks the most relevant result first, then picks others that are different from the first to avoid giving you the same information twice.

In [9]:
mmr_docs = [
    Document(page_content="Melting glaciers in the Arctic are causing sea levels to rise globally.", metadata={"topic": "climate", "impact": "oceans"}),
    Document(page_content="Severe droughts and heatwaves are destroying crops and reducing water supplies.", metadata={"topic": "climate", "impact": "agriculture"}),
    Document(page_content="Ocean acidification is killing coral reefs and harming marine life.", metadata={"topic": "climate", "impact": "biodiversity"}),
    Document(page_content="Wildfires are becoming more frequent and intense due to dry weather conditions.", metadata={"topic": "climate", "impact": "forests"}),
    Document(page_content="Extreme weather patterns like hurricanes are becoming more dangerous every year.", metadata={"topic": "climate", "impact": "weather"})
]

In [10]:
mmr_vector_store = Chroma.from_documents(
    documents=mmr_docs,
    embedding=embeddings,
    collection_name='my_collection'
)

In [11]:
retriever = mmr_vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={'k':3,'lambda_mult':0.5}
)

In [12]:
query = "Causes of global warming"
result = retriever.invoke(query)

for i,doc in enumerate(result):
    print(f"\n------Result {i+1}------")
    print(doc.page_content)


------Result 1------
Global warming is caused by the increase of greenhouse gases in the atmosphere.

------Result 2------
Melting glaciers in the Arctic are causing sea levels to rise globally.

------Result 3------
Severe droughts and heatwaves are destroying crops and reducing water supplies.


# Multi Query Retriever
This method takes one user question and rewrites it in different ways to find better results from a database.

In [13]:
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain_groq import ChatGroq

In [14]:
import os
os.environ['GROQ_API_KEY'] = "your_api_key"

In [15]:
documents = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [16]:
mqr_vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name='new_collection'
)

In [22]:
llm = ChatGroq(model="groq/compound")

similarity_retriever = mqr_vector_store.as_retriever(search_type='similarity',search_kwargs={'k':5})

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=mqr_vector_store.as_retriever(search_kwargs={'k':5}),
    llm=llm
)

In [18]:
result = similarity_retriever.invoke("How to improve energy levels and maintain balance?")

for res in result:
    print(res.page_content)

Drinking sufficient water throughout the day helps maintain metabolism and energy.
The solar energy system in modern homes helps balance electricity demand.
Consuming leafy greens and fruits helps detox the body and improve longevity.
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
Photosynthesis enables plants to produce energy by converting sunlight.


In [23]:
mqr = multi_query_retriever.invoke("Suggest 3 ways to improve energy levels?")

for doc in mqr:
    print(doc.page_content)

Python balances readability with power, making it a popular system design language.
Photosynthesis enables plants to produce energy by converting sunlight.
The solar energy system in modern homes helps balance electricity demand.
Black holes bend spacetime and store immense gravitational energy.
The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.
Consuming leafy greens and fruits helps detox the body and improve longevity.
Regular walking boosts heart health and can reduce symptoms of depression.
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
Drinking sufficient water throughout the day helps maintain metabolism and energy.
Deep sleep is crucial for cellular repair and emotional regulation.


# Contextual Retriever

In [27]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor

In [25]:
cr_docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor and weapons made of metal."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s with silent films.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [26]:
cr_vector_store = Chroma.from_documents(
    documents=cr_docs,
    embedding=embeddings,
    collection_name='cr_collection'
)

In [28]:
cr_retriever = cr_vector_store.as_retriever(search_kwargs={'k':4})

In [29]:
#compressor
compressor = LLMChainExtractor.from_llm(llm)

#contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever = cr_retriever,
    base_compressor = compressor
)

In [30]:
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

for i, doc in enumerate(compressed_results):
    print(f"\n------Result {i+1}------")
    print(doc.page_content)


------Result 1------
**Extracted relevant part (as it appears in the context):**  
"Photosynthesis is the process by which green plants convert sunlight into energy."

**Answer to the question “What is photosynthesis?”**  
Photosynthesis is the biological process by which green plants (and some other organisms) capture sunlight and use its energy to convert carbon dioxide and water into chemical energy, typically in the form of sugars, while releasing oxygen as a by‑product. The extracted sentence from the context succinctly captures this definition: it is the process by which green plants convert sunlight into energy.

------Result 2------
**Extracted relevant part from the context**

> The chlorophyll in plant cells captures sunlight during photosynthesis.

**Answer to the question “What is photosynthesis?”**

Photosynthesis is the biological process by which green plants (and some other organisms) use chlorophyll—the green pigment in their cells—to capture sunlight. The captured li